# Initialization

In [ ]:
!unzip data-analysis-competition-2025.zip

Archive:  data-analysis-competition-2025.zip
  inflating: Test/claude.csv         
  inflating: Test/deepseek.csv       
  inflating: Test/gemini.csv         
  inflating: Test/gpt.csv            
  inflating: Test/grok.csv           
  inflating: Test/perplexity.csv     
  inflating: Train/claude.csv        
  inflating: Train/deepseek.csv      
  inflating: Train/gemini.csv        
  inflating: Train/gpt.csv           
  inflating: Train/grok.csv          
  inflating: Train/perplexity.csv    
  inflating: sample_submission.csv   


In [1]:
!pip install emoji -q
!pip install catboost -q
!pip install deep-translator -q
!pip install langid -q
!pip install lightgbm -q
!pip install sentence_transformers -q
!pip install xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 71.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 11.0 MB/s eta 0:00:00:00:0100:01


# Import Libraries

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import emoji
from deep_translator import GoogleTranslator
import langid
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.preprocessing import LabelEncoder
import torch
from sklearn.utils.class_weight import compute_class_weight
import re
import unicodedata
import xgboost as xgb
import lightgbm as lgb
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel,TrainingArguments, Trainer
from sentence_transformers import SentenceTransformer
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model, TaskType
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_class_weight


tqdm.pandas()

# Read Dataset

In [ ]:
train_dir = 'Train/'
csv_files = [f for f in os.listdir(train_dir) if f.endswith('.csv')]

train_data = {}
for file in csv_files:
    file_path = os.path.join(train_dir, file)
    df_name = os.path.splitext(file)[0]
    train_data[df_name] = pd.read_csv(file_path)
    train_data[df_name]['AI'] = df_name

In [ ]:
test_dir = 'Test/'
csv_files = [f for f in os.listdir(test_dir) if f.endswith('.csv')]

test_data = {}
for file in csv_files:
    file_path = os.path.join(test_dir, file)
    df_name = os.path.splitext(file)[0]

    df = pd.read_csv(file_path)
    df["CommentId"] = df["CommentId"].astype(str).apply(lambda x: f"{df_name}_{x}")
    df["AI"] = df_name
    test_data[df_name] = df

In [ ]:
prefix_order = ['gpt', 'claude', 'deepseek', 'gemini', 'grok', 'perplexity']

combined_train = pd.concat(
    [train_data[prefix] for prefix in prefix_order if prefix in train_data],
    ignore_index=True
)
combined_train.head()

,Comment,At,AppVersion,Sentiment,AI
0,অনেক ভালো একটি এপ আপনার' ও ব্যবহার করুনঃ অনেক ...,2025-06-04 09:50:35,1.2025.147,2,gpt
1,very good at Al and useful,2025-06-16 01:40:52,1.2025.154,2,gpt
2,I love it,2025-06-21 05:59:25,1.2025.154,2,gpt
3,good app,2025-07-02 15:52:41,1.2025.175,2,gpt
4,improve,2025-04-06 09:05:10,1.2025.084,2,gpt


In [ ]:
combined_test = pd.concat(test_data.values(), ignore_index=True)
combined_test.head()

,CommentId,Comment,At,AppVersion,AI
0,perplexity_1,perfect,2025-02-16 10:24:29,2.40.1,perplexity
1,perplexity_2,nice,2025-05-30 03:06:31,2.44.1,perplexity
2,perplexity_3,great work Perplexity Team,2025-02-08 02:32:34,2.39.0,perplexity
3,perplexity_4,তেমন একটা ভালো না টাকা চাই শুধু,2025-06-29 08:24:35,2.48.2,perplexity
4,perplexity_5,too good,2025-05-21 09:04:46,2.44.1,perplexity


In [ ]:
prefix_order = ['gpt', 'claude', 'deepseek', 'gemini', 'grok', 'perplexity']
combined_test['prefix'] = combined_test['CommentId'].apply(lambda x: x.split('_')[0])
combined_test['comment_number'] = combined_test['CommentId'].apply(lambda x: int(x.split('_')[1]))
combined_test['prefix'] = pd.Categorical(combined_test['prefix'], categories=prefix_order, ordered=True)
combined_test = combined_test.sort_values(by=['prefix', 'comment_number']).drop(columns=['prefix', 'comment_number'])
combined_test.head()

,CommentId,Comment,At,AppVersion,AI
26080,gpt_1,super bussniss idea 💡 for chat gpt birrlant ap,2025-04-07 15:27:56,1.2025.084,gpt
26081,gpt_2,good,2025-04-30 14:54:05,1.2025.105,gpt
26082,gpt_3,Very nice,2025-04-26 06:44:56,1.2025.105,gpt
26083,gpt_4,could be more free,2025-06-07 22:44:57,1.2025.147,gpt
26084,gpt_5,very good very good very Nice,2025-04-16 08:30:34,1.2025.091,gpt


# Feature Engineering

We detected the language of each comment in the dataset and translated all non-English text into English. By unifying the text, we reduce variability caused by multilingual inputs and make the data more consistent for downstream tasks such as sentiment analysis. Unfortunately, some languages could not be translated due to tool limitations, so those comments remain in their original form.

In [ ]:
def detect_and_translate(text):
    try:
        lang, _ = langid.classify(text)
    except:
        return text, "unknown"

    if lang != "en":
        try:
            text = GoogleTranslator(source="auto", target="en").translate(text)
        except:
            pass
    return text, lang

In [ ]:
train = combined_train.copy()
train[["Comment_translated", "language"]] = combined_train["Comment"].progress_apply(
    lambda x: detect_and_translate(x) if isinstance(x, str) else (x, "unknown")
).apply(pd.Series)

In [ ]:
test = combined_test.copy()
test[["Comment_translated", "language"]] = combined_test["Comment"].progress_apply(
    lambda x: detect_and_translate(x) if isinstance(x, str) else (x, "unknown")
).apply(pd.Series)

AppVersion preprocessing

- Fill missing values using forward-fill and backward-fill within each AI group.

- Convert version strings into numeric tuples (major, minor, patch) for proper ordering.

- Assign ordinal encoding per AI (older versions → smaller numbers, newer versions → larger numbers).

- Handle unseen versions by assigning them the index of the latest version.

Comment preprocessing

- Handle missing comments by dropping or replacing with defaults.

- Use the translated comment if available, otherwise fall back to the original.

- Convert emojis into descriptive text (e.g., 😀 → :grinning_face:).

In [4]:
train = pd.read_csv('/kaggle/input/prs-its/new_train_detect_and_translate.csv')
test = pd.read_csv('/kaggle/input/prs-its/new_test_detect_and_translate.csv')

In [5]:
def parse_version(v):
    if pd.isna(v):
        return (0, 0, 0)
    parts = v.split(".")
    clean_parts = []
    for p in parts:
        m = re.match(r"(\d+)", p)
        if m:
            clean_parts.append(int(m.group(1)))
        else:
            clean_parts.append(0)
    clean_parts = clean_parts + [0]*(3-len(clean_parts))
    return tuple(clean_parts[:3])

#train
filled = (
    train.groupby("AI", group_keys=False)
         .apply(lambda g: g.sort_values("At")["AppVersion"].ffill().bfill())
         .reindex(train.index)   # balikin ke urutan asli
)

train["AppVersion"] = filled

# buat tuple versi
train["Version_tuple"] = train["AppVersion"].apply(parse_version)

mapping = {}

for ai, subdf in train.groupby("AI"):
    # unique version -> rank
    rank = (
        subdf["Version_tuple"]
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
        .reset_index()
        .set_index("Version_tuple")["index"]
    )
    mapping[ai] = rank.to_dict()

def encode_version(ai, version):
    if ai in mapping and version in mapping[ai]:
        return mapping[ai][version]
    else:
        return len(mapping[ai])  # unseen version diberi asumsi versi terbaru

train["AppVersion"] = train.apply(
    lambda r: encode_version(r["AI"], r["Version_tuple"]), axis=1
)

train = train.dropna(subset=['Comment']).reset_index(drop=True)
train['Comment_translated'] = train['Comment_translated'].fillna(train['Comment'])
train['Comment_translated'] = train['Comment_translated'].apply(lambda x: emoji.demojize(x))
train.drop(columns=['Comment', 'language','Version_tuple'],inplace=True)
train.rename(columns={'Comment_translated': 'Comment'}, inplace=True)

# test
filled = (
    test.groupby("AI", group_keys=False)
         .apply(lambda g: g.sort_values("At")["AppVersion"].ffill().bfill())
         .reindex(test.index)   # balikin ke urutan asli
)

test["AppVersion"] = filled
# buat tuple versi
test["Version_tuple"] = test["AppVersion"].apply(parse_version)

test["AppVersion"] = test.apply(
    lambda r: encode_version(r["AI"], r["Version_tuple"]), axis=1
)

test['Comment'] = test['Comment'].fillna('Good')
test['Comment_translated'] = test['Comment_translated'].fillna(test['Comment'])
test['Comment_translated'] = test['Comment_translated'].apply(lambda x: emoji.demojize(x))
test.drop(columns=['Comment', 'language','Version_tuple'],inplace=True)
test.rename(columns={'Comment_translated': 'Comment'}, inplace=True)

/tmp/ipykernel_36/1261442904.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sort_values("At")["AppVersion"].ffill().bfill())
/tmp/ipykernel_36/1261442904.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sort_values("At")["AppVersion"].ffill().bfill())


During data cleaning, we found that the dataset contained a surprisingly large number of duplicate rows. To ensure data quality and prevent the model from being biased by repeated samples, we removed these duplicates.

In [6]:
# Find duplicate comments
duplicated_comments_mask = train.duplicated(subset=['Comment'], keep=False)
duplicated_comments_df = train[duplicated_comments_mask]

# Find unique comments
unique_comments_df = train[~duplicated_comments_mask].copy()

if not duplicated_comments_df.empty:
    most_frequent_sentiment = duplicated_comments_df.groupby('Comment')['Sentiment'].agg(lambda x: x.mode()[0]).reset_index()

    merged_df = pd.merge(most_frequent_sentiment, duplicated_comments_df, on=['Comment', 'Sentiment'], how='left')
    cleaned_duplicated_df = merged_df.drop_duplicates(subset=['Comment']).copy()

    train_cleaned = pd.concat([unique_comments_df, cleaned_duplicated_df], ignore_index=True)
else:
    train_cleaned = train.copy()

print(f"Original number of rows: {len(train)}")
print(f"Number of rows after removing duplicate comments based on most frequent sentiment: {len(train_cleaned)}")

train = train_cleaned

duplicate_comments_after = train.duplicated(subset=['Comment']).sum()
print(f"Number of duplicate comments after cleaning: {duplicate_comments_after}")

Original number of rows: 130880
Number of rows after removing duplicate comments based on most frequent sentiment: 77813
Number of duplicate comments after cleaning: 0


Some comments contained unusual or non-standard characters (e.g. special symbols, or visually similar Unicode variants). To standardize them, we applied Unicode NFKC normalization.

In [7]:
def normalize_text(text: str) -> str:
    # 1. NFKC normalization
    text = unicodedata.normalize("NFKC", text)

    text = re.sub(r"[\u200B-\u200F\u202A-\u202E\u2060-\u206F]", "", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip().lower()

    return text

train['Comment'] = train['Comment'].apply(normalize_text)
test['Comment'] = test['Comment'].apply(normalize_text)

To represent each comment in a numerical form suitable for machine learning, we embedded the Comment column using a pretrained language model.

In [8]:
print("Memuat semua model... (hanya sekali)")
embedding_e5 = SentenceTransformer('intfloat/multilingual-e5-large')
embedding_tabularisai = SentenceTransformer('tabularisai/multilingual-sentiment-analysis')
embedding_qwen = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B')
print("Semua model dimuat.")
print("\n" + "="*50 + "\n")

Memuat semua model... (hanya sekali)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Semua model dimuat.




In [9]:
train_df = train.copy()
test_df = test.copy()

In [10]:
def create_features(df, embedding_model, train_columns=None):

    # --- Embedding Text ---
    text_embeddings = embedding_model.encode(df['Comment'].tolist(), show_progress_bar=True)
    embedding_df = pd.DataFrame(text_embeddings, columns=[f'embed_{i}' for i in range(text_embeddings.shape[1])])

    if train_columns is not None:
        categorical_features_df = categorical_features_df.reindex(columns=train_columns, fill_value=0)

    df.reset_index(drop=True, inplace=True)

    return embedding_df

train_e5 = create_features(train_df, embedding_e5)
test_e5 = create_features(test_df, embedding_e5)

train_e5['Sentiment'] = train['Sentiment']

Batches:   0%|          | 0/2432 [00:00<?, ?it/s]

Batches:   0%|          | 0/1437 [00:00<?, ?it/s]

In [11]:
def create_features(df, embedding_model, train_columns=None):

    # --- Embedding Text ---
    text_embeddings = embedding_model.encode(df['Comment'].tolist(), show_progress_bar=True, batch_size=8)
    embedding_df = pd.DataFrame(text_embeddings, columns=[f'embed_{i}' for i in range(text_embeddings.shape[1])])

    if train_columns is not None:
        categorical_features_df = categorical_features_df.reindex(columns=train_columns, fill_value=0)

    df.reset_index(drop=True, inplace=True)

    return embedding_df

In [12]:
train_qwen = create_features(train_df, embedding_qwen)
test_qwen = create_features(test_df, embedding_qwen)

train_qwen['Sentiment'] = train['Sentiment']

Batches:   0%|          | 0/9727 [00:00<?, ?it/s]

Batches:   0%|          | 0/5747 [00:00<?, ?it/s]

In [13]:
def create_features(df, embedding_model, train_columns=None):

    # --- Embedding Text ---
    text_embeddings = embedding_model.encode(df['Comment'].tolist(), show_progress_bar=True)
    embedding_df = pd.DataFrame(text_embeddings, columns=[f'embed_{i}' for i in range(text_embeddings.shape[1])])

    if train_columns is not None:
        categorical_features_df = categorical_features_df.reindex(columns=train_columns, fill_value=0)

    df.reset_index(drop=True, inplace=True)

    return embedding_df

In [14]:
train_tabularisai = create_features(train_df, embedding_tabularisai)
test_tabularisai = create_features(test_df, embedding_tabularisai)

train_tabularisai['Sentiment'] = train['Sentiment']

Batches:   0%|          | 0/2432 [00:00<?, ?it/s]

Batches:   0%|          | 0/1437 [00:00<?, ?it/s]

In [16]:
# Initialize tokenizer and model outside the function
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model = AutoModel.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest", output_hidden_states=True)

def roberta_embeddings(texts, batch_size=32, device="cuda"):
    model.to(device) # Move model to the specified device

    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=512)

        # Ensure inputs are on the correct device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        try:
            with torch.no_grad():
                outputs = model(**inputs)
                # hidden_states[-1] = last layer (batch, seq_len, hidden_dim)
                last_hidden = outputs.last_hidden_state
                # mean pooling across tokens
                embeddings = last_hidden.mean(dim=1)

            all_embeddings.append(embeddings.cpu())
        except Exception as e:
            print(f"Error processing batch {i} to {i+batch_size}: {e}")
            raise

    return torch.cat(all_embeddings, dim=0).numpy()

def create_features(df, train_columns=None, batch_size=8, device="cuda"):
    """
    Create features with text embeddings + categorical one-hot encoding.
    """
    df = df.copy()

    # --- Text Embedding ---
    text_embeddings = roberta_embeddings(df['Comment'].tolist(), batch_size=batch_size, device=device)
    embedding_df = pd.DataFrame(text_embeddings, columns=[f'embed_{i}' for i in range(text_embeddings.shape[1])])


    # --- Combine ---
    df.reset_index(drop=True, inplace=True)

    return embedding_df

train_roberta = create_features(train_df)
test_roberta  = create_features(test_df)

train_roberta['Sentiment'] = train_df['Sentiment']

Encoding: 100%|██████████| 5747/5747 [02:21<00:00, 40.62it/s]


# Modeling

In [22]:
# =====================================================
# ANN Model
# =====================================================
class ANNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.layers(x)


def train_and_evaluate(train_df, test_size=0.2, epochs=10, batch_size=32):
    # 1. Split Data
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_df.drop(columns=['Sentiment']),
        train_df["Sentiment"].tolist(),
        test_size=test_size,
        random_state=42,
        stratify=train_df["Sentiment"]
    )

    X_train = train_texts
    X_val = val_texts
    y_train = np.array(train_labels)
    y_val = np.array(val_labels)

    # 2. Class Weights
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weights_dict = {c: w for c, w in zip(classes, class_weights)}
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
    print("Class weights:", class_weights_dict)

    # =====================================================
    # 1. CatBoost
    # =====================================================
    clf_cat = CatBoostClassifier(
        iterations=200,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiClass",
        class_weights=class_weights_dict,
        task_type="GPU" if torch.cuda.is_available() else "CPU",
        verbose=0
    )
    clf_cat.fit(X_train, y_train, eval_set=(X_val, y_val))
    y_pred_cat = clf_cat.predict(X_val)
    print("\n=== CatBoost ===")
    print(classification_report(y_val, y_pred_cat, target_names=["Negative","Neutral","Positive"]))

    # =====================================================
    # 2. XGBoost
    # =====================================================
    sample_weights_train = np.array([class_weights_dict[label] for label in y_train])
    sample_weights_val = np.array([class_weights_dict[label] for label in y_val])

    clf_xgb = xgb.XGBClassifier(
        objective="multi:softmax",
        num_class=len(classes),
        eval_metric="mlogloss",
        tree_method="gpu_hist" if torch.cuda.is_available() else "hist",
        learning_rate=0.05,
        max_depth=8,
        n_estimators=200,
        verbosity=0
    )
    clf_xgb.fit(
        X_train, y_train,
        sample_weight=sample_weights_train,
        eval_set=[(X_val, y_val)],
        sample_weight_eval_set=[sample_weights_val],
        verbose=False
    )
    y_pred_xgb = clf_xgb.predict(X_val)
    print("\n=== XGBoost ===")
    print(classification_report(y_val, y_pred_xgb, target_names=["Negative","Neutral","Positive"]))

    # =====================================================
    # 3. LightGBM
    # =====================================================
    clf_lgb = lgb.LGBMClassifier(
        boosting_type="gbdt",
        objective="multiclass",
        num_class=len(classes),
        n_estimators=200,
        learning_rate=0.05,
        max_depth=8,
        class_weight=class_weights_dict,
        device="gpu" if torch.cuda.is_available() else "cpu"
    )
    clf_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    y_pred_lgb = clf_lgb.predict(X_val)
    print("\n=== LightGBM ===")
    print(classification_report(y_val, y_pred_lgb, target_names=["Negative","Neutral","Positive"]))

    # =====================================================
    # 4. ANN (PyTorch)
    # =====================================================
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_dim = X_train.shape[1]
    num_classes = len(classes)

    model = ANNClassifier(input_dim, hidden_dim=256, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

    # DataLoader
    train_dataset = TensorDataset(torch.tensor(X_train.values, dtype=torch.float32),
                                  torch.tensor(y_train, dtype=torch.long))
    val_dataset = TensorDataset(torch.tensor(X_val.values, dtype=torch.float32),
                                torch.tensor(y_val, dtype=torch.long))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(yb.cpu().numpy())

    print("\n=== ANN ===")
    print(classification_report(all_labels, all_preds, target_names=["Negative","Neutral","Positive"]))

    return {
        "catboost": clf_cat,
        "xgboost": clf_xgb,
        "lightgbm": clf_lgb,
        "ann": model
    }


## multilingual-e5-large

In [23]:
models = train_and_evaluate(train_e5, epochs=10, batch_size=32)

Class weights: {0: 1.755350647153371, 1: 6.818928688793953, 2: 0.4378930485797493}



=== CatBoost ===
              precision    recall  f1-score   support

    Negative       0.76      0.81      0.78      2955
     Neutral       0.20      0.42      0.27       761
    Positive       0.96      0.88      0.92     11847

    accuracy                           0.84     15563
   macro avg       0.64      0.70      0.66     15563
weighted avg       0.89      0.84      0.86     15563


=== XGBoost ===
              precision    recall  f1-score   support

    Negative       0.77      0.86      0.81      2955
     Neutral       0.27      0.19      0.23       761
    Positive       0.95      0.94      0.94     11847

    accuracy                           0.89     15563
   macro avg       0.66      0.66      0.66     15563
weighted avg       0.88      0.89      0.88     15563

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 261120
[LightGBM] [Info] Number of data points in the train set: 62250, number of used features: 1024
[LightGBM] [Info] Using GPU 

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1024 dense feature groups (60.79 MB) transferred to GPU in 0.038396 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612

=== LightGBM ===
              precision    recall  f1-score   support

    Negative       0.76      0.84      0.80      2955
     Neutral       0.22      0.35      0.27       761
    Positive       0.96      0.90      0.93     11847

    accuracy                           0.86     15563
   macro avg       0.65      0.70      0.67     15563
weighted avg       0.89      0.86      0.87     15563


=== ANN ===
              precision    recall  f1-score   support

    Negative       0.77      0.84      0.81      2955
     Neutral       0.21      0.42      0.28       761
    Positive       0.96      0.88      0.92     118

## Qwen/Qwen3-Embedding-0.6B

In [24]:
models = train_and_evaluate(train_qwen, epochs=10, batch_size=32)

Class weights: {0: 1.755350647153371, 1: 6.818928688793953, 2: 0.4378930485797493}



=== CatBoost ===
              precision    recall  f1-score   support

    Negative       0.70      0.78      0.74      2955
     Neutral       0.18      0.37      0.24       761
    Positive       0.95      0.86      0.90     11847

    accuracy                           0.82     15563
   macro avg       0.61      0.67      0.63     15563
weighted avg       0.87      0.82      0.84     15563


=== XGBoost ===
              precision    recall  f1-score   support

    Negative       0.71      0.82      0.76      2955
     Neutral       0.24      0.15      0.18       761
    Positive       0.94      0.92      0.93     11847

    accuracy                           0.87     15563
   macro avg       0.63      0.63      0.62     15563
weighted avg       0.86      0.87      0.86     15563

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 261120
[LightGBM] [Info] Number of data points in the train set: 62250, number of used features: 1024
[LightGBM] [Info] Using GPU 

## tabularisai/multilingual-sentiment-analysis



In [25]:
models = train_and_evaluate(train_tabularisai, epochs=10, batch_size=32)

Class weights: {0: 1.755350647153371, 1: 6.818928688793953, 2: 0.4378930485797493}



=== CatBoost ===
              precision    recall  f1-score   support

    Negative       0.68      0.76      0.72      2955
     Neutral       0.18      0.45      0.26       761
    Positive       0.96      0.84      0.89     11847

    accuracy                           0.80     15563
   macro avg       0.61      0.68      0.62     15563
weighted avg       0.87      0.80      0.83     15563


=== XGBoost ===
              precision    recall  f1-score   support

    Negative       0.71      0.81      0.76      2955
     Neutral       0.23      0.26      0.24       761
    Positive       0.94      0.90      0.92     11847

    accuracy                           0.85     15563
   macro avg       0.63      0.66      0.64     15563
weighted avg       0.86      0.85      0.86     15563

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 62250, number of used features: 768
[LightGBM] [Info] Using GPU D

## cardiffnlp/twitter-roberta-base-sentiment-latest

In [26]:
models = train_and_evaluate(train_roberta, epochs=10, batch_size=32)

Class weights: {0: 1.755350647153371, 1: 6.818928688793953, 2: 0.4378930485797493}



=== CatBoost ===
              precision    recall  f1-score   support

    Negative       0.73      0.78      0.75      2955
     Neutral       0.20      0.43      0.27       761
    Positive       0.96      0.87      0.91     11847

    accuracy                           0.83     15563
   macro avg       0.63      0.69      0.64     15563
weighted avg       0.88      0.83      0.85     15563


=== XGBoost ===
              precision    recall  f1-score   support

    Negative       0.75      0.83      0.79      2955
     Neutral       0.24      0.25      0.25       761
    Positive       0.95      0.92      0.93     11847

    accuracy                           0.87     15563
   macro avg       0.65      0.67      0.66     15563
weighted avg       0.88      0.87      0.87     15563

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 62250, number of used features: 768
[LightGBM] [Info] Using GPU D

## Submission

In [ ]:
X = train_e5.drop(columns=['Sentiment'])
y = train_e5["Sentiment"].values

final_clf = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    class_weights=class_weights_dict,
    task_type="GPU",
    verbose=100
)

final_clf.fit(X, y, verbose=False)

X_test = test_e5

y_test_pred = final_clf.predict(X_test)

test_e5["Predict"] = y_test_pred.astype(int)

In [ ]:
# --- Make Submission ---
test_df['CommentId'] = test['CommentId']
submission = test_df[["CommentId", "Predict"]].copy()
submission = submission.rename(columns={"Predict": "Sentiment"})

# --- Save CSV ---
submission.to_csv("submission.csv", index=False)

submission.head()

,CommentId,Sentiment
0,gpt_1,2
1,gpt_2,2
2,gpt_3,2
3,gpt_4,1
4,gpt_5,2
